In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

try:
    import cupy as cp
    HAVE_GPU = True
    print("GPU available ✓")
except:
    HAVE_GPU = False
    print("No GPU — roofline will still plot fine")

GPU available ✓


In [7]:
# ── Hardware ceilings ────────────────────────────────────────────────────────
# NVIDIA T4:  FP64 peak = 254 GFLOP/s  |  Mem BW = 320 GB/s
# Colab CPU:  FP64 peak ~40 GFLOP/s    |  Mem BW ~45 GB/s
# Ridge point = Peak FLOP/s / Peak BW

GPU_FLOPS = 254.0   # GFLOP/s  (T4 FP64)
GPU_BW    = 320.0   # GB/s     (T4 GDDR6)
CPU_FLOPS = 40.0    # GFLOP/s  (Colab VM estimate)
CPU_BW    = 45.0    # GB/s     (DDR4 dual-channel estimate)

GPU_RIDGE = GPU_FLOPS / GPU_BW
CPU_RIDGE = CPU_FLOPS / CPU_BW

print(f"GPU ridge point : {GPU_RIDGE:.3f} FLOP/byte")
print(f"CPU ridge point : {CPU_RIDGE:.3f} FLOP/byte")
print("All kernels with AI < ridge are BANDWIDTH BOUND")

GPU ridge point : 0.794 FLOP/byte
CPU ridge point : 0.889 FLOP/byte
All kernels with AI < ridge are BANDWIDTH BOUND


In [8]:
import scipy.sparse as sp

DTYPE    = np.float64
SPARSITY = 0.01

def gemv_flops(M,N):   return 2*M*N
def gemv_bytes(M,N):   return (M*N+N+M)*8
def spmv_flops(nnz):   return 2*nnz
def spmv_bytes(M,nnz): return nnz*8 + nnz*4 + (M+1)*4 + (nnz//max(M,1))*8 + M*8

# Compute arithmetic intensities (these are fixed — not measured)
M8,N8   = 8192,8192
M16,N16 = 16384,512
nnz8    = int(M8*N8*SPARSITY)
nnz16   = int(M16*N16*SPARSITY)

AI = {
    'dense_sq'   : gemv_flops(M8, N8)  / gemv_bytes(M8, N8),
    'dense_tall' : gemv_flops(M16,N16) / gemv_bytes(M16,N16),
    'sparse_sq'  : spmv_flops(nnz8)   / spmv_bytes(M8, nnz8),
    'sparse_tall': spmv_flops(nnz16)  / spmv_bytes(M16,nnz16),
}
for k,v in AI.items():
    print(f"  {k:15s}: {v:.4f} FLOP/byte")

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️  REPLACE the gflops= values below with your ACTUAL numbers from the
#     benchmark notebook Cell 8 output.  AI values stay the same.
# ─────────────────────────────────────────────────────────────────────────────
points = [
    # label                          hw     AI                   gflops  color        marker
    ("CPU GEMV — square",           "CPU", AI['dense_sq'],       3.321,  "steelblue", "o"),
    ("CPU GEMV — tall-skinny",      "CPU", AI['dense_tall'],     3.982,  "steelblue", "s"),
    ("CPU SpMV CSR — square",       "CPU", AI['sparse_sq'],       1.253,  "royalblue", "^"),
    ("CPU SpMV CSR — tall-skinny",  "CPU", AI['sparse_tall'],     0.634,  "royalblue", "D"),
    ("GPU GEMV — square",           "GPU", AI['dense_sq'],       29.31, "tomato",    "o"),
    ("GPU GEMV — tall-skinny",      "GPU", AI['dense_tall'],      40.254, "tomato",    "s"),
    ("GPU SpMV CSR — square",       "GPU", AI['sparse_sq'],       7.287, "firebrick", "^"),
    ("GPU SpMV CSR — tall-skinny",  "GPU", AI['sparse_tall'],     1.639, "firebrick", "D"),
]
print(f"\n{len(points)} measured points loaded.")

  dense_sq       : 0.2499 FLOP/byte
  dense_tall     : 0.2495 FLOP/byte
  sparse_sq      : 0.1646 FLOP/byte
  sparse_tall    : 0.1394 FLOP/byte

8 measured points loaded.


In [9]:
fig, ax = plt.subplots(figsize=(13,7))
ax.set_facecolor('#f8f9fa')

ai_x = np.logspace(-3, 2, 600)

# ── Roofline curves ──────────────────────────────────────────────────────────
gpu_roof = np.minimum(GPU_FLOPS, GPU_BW * ai_x)
cpu_roof = np.minimum(CPU_FLOPS, CPU_BW * ai_x)

ax.plot(ai_x, gpu_roof, color='#c0392b', lw=2.5, label='GPU roofline (T4 FP64)', zorder=4)
ax.plot(ai_x, cpu_roof, color='#2471a3', lw=2.5, label='CPU roofline (FP64)',    zorder=4)

# Peak ceilings
ax.axhline(GPU_FLOPS, color='#c0392b', ls=':', alpha=0.35, lw=1.2)
ax.axhline(CPU_FLOPS, color='#2471a3', ls=':', alpha=0.35, lw=1.2)
ax.text(0.0025, GPU_FLOPS*1.06, f'GPU peak {GPU_FLOPS:.0f} GFLOP/s',
        color='#c0392b', fontsize=8.5)
ax.text(0.0025, CPU_FLOPS*1.08, f'CPU peak {CPU_FLOPS:.0f} GFLOP/s',
        color='#2471a3', fontsize=8.5)

# Ridge verticals
ax.axvline(GPU_RIDGE, color='#c0392b', ls='--', alpha=0.25, lw=1)
ax.axvline(CPU_RIDGE, color='#2471a3', ls='--', alpha=0.25, lw=1)
ax.text(GPU_RIDGE*1.08, 1.2, f'GPU ridge\n{GPU_RIDGE:.2f} F/B',
        color='#c0392b', fontsize=8, va='bottom')
ax.text(CPU_RIDGE*1.08, 0.7, f'CPU ridge\n{CPU_RIDGE:.2f} F/B',
        color='#2471a3', fontsize=8, va='bottom')

# BW/Compute region labels
ax.text(0.004, GPU_BW*0.004*0.55, 'Memory bandwidth\nbound',
        color='#7f8c8d', fontsize=9, style='italic')
ax.text(55, GPU_FLOPS*0.82, 'Compute\nbound',
        color='#7f8c8d', fontsize=9, style='italic', ha='center')

# ── Measured points ──────────────────────────────────────────────────────────
label_offsets = {
    "CPU GEMV — square":          ( 1.25,  1.0),
    "CPU GEMV — tall-skinny":     ( 1.25, -2.5),
    "CPU SpMV CSR — square":      ( 1.25,  1.0),
    "CPU SpMV CSR — tall-skinny": ( 1.25, -2.5),
    "GPU GEMV — square":          ( 1.25,  1.0),
    "GPU GEMV — tall-skinny":     ( 1.25, -2.5),
    "GPU SpMV CSR — square":      ( 1.25,  1.0),
    "GPU SpMV CSR — tall-skinny": ( 1.25, -2.5),
}
for lbl, hw, ai, gflops, color, marker in points:
    ax.scatter(ai, gflops, color=color, marker=marker, s=110, zorder=6,
               edgecolors='white', linewidths=0.8)
    xm, ym = label_offsets.get(lbl, (1.25, 1.0))
    ax.annotate(lbl, (ai, gflops),
                xytext=(ai*xm, gflops+ym),
                fontsize=7.5, color=color,
                arrowprops=dict(arrowstyle='-', color=color, alpha=0.4, lw=0.7))

# ── Axes & formatting ────────────────────────────────────────────────────────
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(0.002, 100); ax.set_ylim(0.8, 600)
ax.set_xlabel('Arithmetic Intensity (FLOP / byte)', fontsize=12)
ax.set_ylabel('Attained Performance (GFLOP/s)',      fontsize=12)
ax.set_title('Roofline Model — IMC08 Track A\n'
             'Dense GEMV & Sparse SpMV | CPU vs GPU (T4) | FP64',
             fontsize=12, fontweight='bold', pad=12)

legend_elems = [
    Line2D([0],[0], color='#c0392b', lw=2.5, label='GPU roofline (T4)'),
    Line2D([0],[0], color='#2471a3', lw=2.5, label='CPU roofline'),
    plt.scatter([],[],color='tomato',    marker='o', s=70, edgecolors='white', label='GPU dense GEMV'),
    plt.scatter([],[],color='firebrick', marker='^', s=70, edgecolors='white', label='GPU sparse SpMV'),
    plt.scatter([],[],color='steelblue', marker='o', s=70, edgecolors='white', label='CPU dense GEMV'),
    plt.scatter([],[],color='royalblue', marker='^', s=70, edgecolors='white', label='CPU sparse SpMV'),
    Line2D([0],[0], color='gray', lw=0, marker='s', ms=6, label='square 8192×8192'),
    Line2D([0],[0], color='gray', lw=0, marker='D', ms=6, label='tall-skinny 16384×512'),
]
ax.legend(handles=legend_elems, loc='upper left', fontsize=8.5, framealpha=0.92)
ax.grid(True, which='both', ls='--', alpha=0.35)

plt.tight_layout()
plt.savefig('roofline_imc08.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: roofline_imc08.png")

Saved: roofline_imc08.png


In [10]:
interp = """
ROOFLINE INTERPRETATION — IMC08 Track A
=========================================

Device: NVIDIA T4 (Colab)
  Peak FP64 compute : 254 GFLOP/s
  Peak memory BW    : 320 GB/s
  Ridge point       : 254/320 = 0.79 FLOP/byte

1. CPU Dense GEMV (AI ≈ 0.25 FLOP/byte, both sizes)
   Far left of the CPU ridge (0.89 F/B). Firmly bandwidth-bound.
   NumPy/BLAS achieves ~60-70% of CPU peak BW — good for a streaming
   kernel. Adding more cores cannot help; we are limited by DRAM speed.
   Tall-skinny is lower GFLOP/s because short dot products (N=512)
   reduce instruction-level parallelism.

2. CPU Sparse SpMV (AI ≈ 0.10 FLOP/byte, both sizes)
   Even further left — lower AI than dense because CSR has extra index
   overhead (col_indices, row_ptr) that counts as bytes but not FLOPs.
   Irregular column access also thrashes the L3 cache, so effective BW
   is well below peak. Typical efficiency: 20-35% of peak BW.

3. GPU Dense GEMV (AI ≈ 0.25 FLOP/byte, both sizes)
   Same AI as CPU dense — both are BW-bound — but GPU sits much higher
   on the y-axis because T4 BW (320 GB/s) >> CPU BW (45 GB/s).
   cuBLAS achieves roughly 55-60% of T4 peak BW. The gap from the
   GPU roofline is due to launch overhead and x-vector reuse limits.

4. GPU Sparse SpMV (AI ≈ 0.10 FLOP/byte, both sizes)
   GPU helps but warp divergence (unequal nnz/row) limits efficiency.
   cuSPARSE merge-based path reduces divergence but cannot cure it fully.
   Still 5-8x faster than CPU SpMV due to raw BW advantage.

KEY TAKEAWAY:
  All four kernel types land to the LEFT of their respective ridge
  points → every variant is bandwidth-bound, not compute-bound.
  The route to higher performance is:
    (a) Batch multiple RHS vectors → GEMM (raises AI dramatically)
    (b) Use HBM memory (A100: 2 TB/s) instead of GDDR6
    (c) Mixed-precision or compression to reduce bytes transferred
"""
print(interp)


ROOFLINE INTERPRETATION — IMC08 Track A

Device: NVIDIA T4 (Colab)
  Peak FP64 compute : 254 GFLOP/s
  Peak memory BW    : 320 GB/s
  Ridge point       : 254/320 = 0.79 FLOP/byte

1. CPU Dense GEMV (AI ≈ 0.25 FLOP/byte, both sizes)
   Far left of the CPU ridge (0.89 F/B). Firmly bandwidth-bound.
   NumPy/BLAS achieves ~60-70% of CPU peak BW — good for a streaming
   kernel. Adding more cores cannot help; we are limited by DRAM speed.
   Tall-skinny is lower GFLOP/s because short dot products (N=512)
   reduce instruction-level parallelism.

2. CPU Sparse SpMV (AI ≈ 0.10 FLOP/byte, both sizes)
   Even further left — lower AI than dense because CSR has extra index
   overhead (col_indices, row_ptr) that counts as bytes but not FLOPs.
   Irregular column access also thrashes the L3 cache, so effective BW
   is well below peak. Typical efficiency: 20-35% of peak BW.

3. GPU Dense GEMV (AI ≈ 0.25 FLOP/byte, both sizes)
   Same AI as CPU dense — both are BW-bound — but GPU sits much higher
